In [21]:
from pathlib import Path
from collections.abc import Sequence

import cv2
import numpy as np
import pandas as pd
from scipy import stats
from skimage.feature import graycomatrix, graycoprops

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


# 1 预处理 Proprocessing

预处理工作流分为三步：
1. 颜色矫正：对于 RGB 图像进行如下操作
    - 计算每个通道的 $L_c$ 得到估计出来的环境光照 $ \text{Light} = (L_r, L_g, L_b) $ $$ L_c = \sqrt[p]{\frac 1 N \cdot \sum_{x, y} I_c(x, y)^p} $$
    - 计算每个通道的增益系数 gain $$ \text{gain} = \left(\frac{\max(L_r, L_g, L_b)}{L_r}, \frac{\max(L_r, L_g, L_b)}{L_g}, \frac{\max(L_r, L_g, L_b)}{L_b} \right)$$
    - 将该图像的每个像素 pixel 都通过以上增益
2. 中值滤波：去除椒盐噪声
3. 毛发祛除：使用 Dull-Razor 方法对图像中的暗色毛发区域进行检测与修复
    - 将输入图像转换为灰度图 $G(x, y)$，减少颜色通道对毛发检测的影响
    - 使用形态学黑帽变换突出比周围皮肤更暗、更细长的毛发结构 $$ B = \text{BlackHat}(G, K) = \text{Close}(G, K) - G $$ 其中 $K$ 为十字形结构元素
    - 对黑帽图像进行阈值分割，得到毛发区域的二值掩膜 $$ M(x, y) = \begin{cases}255, & B(x, y) > T \\ 0, & B(x, y) \le T\end{cases} $$
    - 根据毛发掩膜 $M$ 使用 Telea 图像修复方法对原图进行 inpaint，使用周围皮肤像素估计并填补毛发区域


In [2]:
def shades_of_gray(img: np.ndarray, p: int = 6) -> np.ndarray:
    """Shades of Gray color constancy."""
    img_float = img.astype(np.float32) / 255.0
    b, g, r = cv2.split(img_float)

    l_b = np.power(np.mean(np.power(b, p)), 1.0 / p)
    l_g = np.power(np.mean(np.power(g, p)), 1.0 / p)
    l_r = np.power(np.mean(np.power(r, p)), 1.0 / p)

    if l_b == 0 or l_g == 0 or l_r == 0:
        return img.copy()

    l_max = np.max([l_b, l_g, l_r])
    img_corrected = cv2.merge([
        b * (l_max / l_b) * 255.0,
        g * (l_max / l_g) * 255.0,
        r * (l_max / l_r) * 255.0,
    ])

    return np.clip(img_corrected, 0, 255).astype(np.uint8)


def dull_razor_hair_removal(
    img: np.ndarray,
    kernel_size: int = 10,
    threshold: int = 10,
    inpaint_radius: int = 1,
) -> np.ndarray:
    """Remove hair-like dark structures with the Dull-Razor method."""
    gray_scale = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray_scale, cv2.MORPH_BLACKHAT, kernel)
    _, hair_mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)

    return cv2.inpaint(img, hair_mask, inpaint_radius, cv2.INPAINT_TELEA)


def pipeline_preview(
    input_dir: str | Path,
    output_dir: str | Path,
    progress_interval: int = 100,
    color_power: int = 6,
    median_kernel_size: int = 3,
    hair_kernel_size: int = 10,
    hair_threshold: int = 10,
    inpaint_radius: int = 1,
) -> None:
    """Process jpg images and save only the final preprocessed results."""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(input_path.glob('*.jpg'))
    total_images = len(image_paths)

    if total_images == 0:
        print(f"[ERROR] 在文件夹 '{input_path}' 中没有找到 jpg 图片。")
        return

    print(f"[SUCCESS] 成功检索到 {total_images} 张 jpg 图片，开始批量处理...")

    processed_count = 0
    failed_files: list[str] = []

    for index, image_path in enumerate(image_paths, start=1):
        img = cv2.imread(str(image_path))
        if img is None:
            failed_files.append(image_path.name)
            continue

        try:
            img_cc = shades_of_gray(img, p=color_power)
            img_blur = cv2.medianBlur(img_cc, median_kernel_size)
            img_final = dull_razor_hair_removal(
                img_blur,
                kernel_size=hair_kernel_size,
                threshold=hair_threshold,
                inpaint_radius=inpaint_radius,
            )
            cv2.imwrite(str(output_path / image_path.name), img_final)
            processed_count += 1
        except Exception:
            failed_files.append(image_path.name)
            continue

        if index % progress_interval == 0 or index == total_images:
            print(f"[PROGRESS] 已处理 {index}/{total_images} 张图片")

    print(f"[SUCCESS] 批量处理完成，已保存 {processed_count} 张图片至: {output_path}")
    if failed_files:
        print(f"[WARNING] 跳过 {len(failed_files)} 张读取失败或处理失败的图片。")


In [4]:
# test cell
PROJECT_DIR = Path('./')
INPUT_DIR = PROJECT_DIR / 'image'
OUTPUT_DIR = PROJECT_DIR / 'image_processed'

pipeline_preview(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    progress_interval=100,
)

[SUCCESS] 成功检索到 600 张 jpg 图片，开始批量处理...
[PROGRESS] 已处理 100/600 张图片
[PROGRESS] 已处理 200/600 张图片
[PROGRESS] 已处理 300/600 张图片
[PROGRESS] 已处理 400/600 张图片
[PROGRESS] 已处理 500/600 张图片
[PROGRESS] 已处理 600/600 张图片
[SUCCESS] 批量处理完成，已保存 600 张图片至: image_processed


# 2 特征提取 Feature extraction

特征提取部分主要完成 4 个部分的特征提取：
1. 形态特征 Shape
2. 抽象特征 / 不对称特征
3. 颜色特征（8 个空间）
4. 纹理特征（GLCM）

## 2.1 形态特征 & 不对称特征

In [7]:
# functions for Shape features and asymmetry features

def _to_binary_mask(mask: np.ndarray) -> np.ndarray:
    if mask.ndim == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    return binary


def extract_lesion_feature_shape(
    images: Sequence[np.ndarray],
    masks: Sequence[np.ndarray],
    eps: float = 1e-8,
) -> tuple[np.ndarray, list[str]]:
    """Extract lesion shape features from masks."""
    feature_rows: list[list[float]] = []
    feature_names = [
        "shape_area",
        "shape_perimeter",
        "shape_circularity",
        "shape_max_diameter",
        "shape_equiv_diameter",
        "shape_aspect_ratio",
        "shape_eccentricity",
        "shape_compactness",
        "shape_solidity",
        "shape_rectangularity",
        "shape_elongation",
        "shape_defect_ratio",
    ]

    for mask in masks:
        thresh = _to_binary_mask(mask)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if len(contours) == 0 or np.sum(thresh) == 0:
            feature_rows.append([0.0] * len(feature_names))
            continue

        contour = max(contours, key=cv2.contourArea)
        area = float(cv2.contourArea(contour))
        perimeter = float(cv2.arcLength(contour, True))
        circularity = (4.0 * np.pi * area) / (perimeter ** 2 + eps)

        points = contour.reshape(-1, 2)
        if len(points) > 1:
            diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
            max_diameter = float(np.sqrt(np.max(np.sum(diff ** 2, axis=-1))))
        else:
            max_diameter = 0.0

        equiv_diameter = float(np.sqrt(4.0 * area / np.pi)) if area > 0 else 0.0

        if len(contour) >= 5:
            (_, _), (d1, d2), _ = cv2.fitEllipse(contour)
            major_axis = max(d1, d2)
            minor_axis = min(d1, d2)
            aspect_ratio = major_axis / (minor_axis + eps)
            eccentricity = float(np.sqrt(max(0.0, 1.0 - (minor_axis ** 2) / (major_axis ** 2 + eps))))
        else:
            aspect_ratio = 1.0
            eccentricity = 0.0

        compactness = equiv_diameter / (max_diameter + eps)
        hull = cv2.convexHull(contour)
        hull_area = float(cv2.contourArea(hull))
        solidity = area / (hull_area + eps)
        defect_ratio = max(0.0, hull_area - area) / (area + eps)

        (_, _), (rect_w, rect_h), _ = cv2.minAreaRect(contour)
        bbox_w = max(rect_w, rect_h)
        bbox_h = min(rect_w, rect_h)
        rectangularity = area / (bbox_w * bbox_h + eps)
        elongation = bbox_h / (bbox_w + eps)

        feature_rows.append([
            area,
            perimeter,
            circularity,
            max_diameter,
            equiv_diameter,
            aspect_ratio,
            eccentricity,
            compactness,
            solidity,
            rectangularity,
            elongation,
            defect_ratio,
        ])

    return np.asarray(feature_rows, dtype=np.float32), feature_names


def extract_lesion_feature_asymmetry(
    images: Sequence[np.ndarray],
    masks: Sequence[np.ndarray],
    eps: float = 1e-8,
) -> tuple[np.ndarray, list[str]]:
    """Extract lesion asymmetry features from masks."""
    feature_rows: list[list[float]] = []
    feature_names = [
        "asymmetry_area_ratio",
        "asymmetry_x_axis",
        "asymmetry_xy_sum",
        "asymmetry_rotation",
        "asymmetry_fullness",
    ]

    for mask in masks:
        thresh = _to_binary_mask(mask)
        total_area = float(np.sum(thresh == 255))

        if total_area == 0:
            feature_rows.append([0.0] * len(feature_names))
            continue

        moments = cv2.moments(thresh)
        if moments["m00"] != 0:
            cx = int(round(moments["m10"] / moments["m00"]))
            cy = int(round(moments["m01"] / moments["m00"]))
        else:
            cx, cy = thresh.shape[1] // 2, thresh.shape[0] // 2

        diff_x = cv2.absdiff(thresh, cv2.flip(thresh, 1))
        diff_y = cv2.absdiff(thresh, cv2.flip(thresh, 0))
        area_diff_x = float(np.sum(diff_x == 255))
        area_diff_y = float(np.sum(diff_y == 255))

        asymmetry_x_axis = (area_diff_x / total_area) * 100.0
        asymmetry_xy_sum = ((area_diff_x + area_diff_y) / total_area) * 100.0

        left_area = float(np.sum(thresh[:, :cx] == 255))
        right_area = float(np.sum(thresh[:, cx:] == 255))
        asymmetry_area_ratio = (abs(left_area - right_area) / total_area) * 100.0

        h, w = thresh.shape
        rot_mat = cv2.getRotationMatrix2D((cx, cy), 180, 1.0)
        rotated_mask = cv2.warpAffine(thresh, rot_mat, (w, h), flags=cv2.INTER_NEAREST)
        fs_mask = cv2.bitwise_xor(thresh, rotated_mask)
        a_mask = cv2.bitwise_or(thresh, rotated_mask)
        asymmetry_rotation = 1.0 - (float(np.sum(fs_mask == 255)) / (float(np.sum(a_mask == 255)) + eps))

        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) > 0:
            contour = max(contours, key=cv2.contourArea)
            if len(contour) >= 5:
                (_, _), (d1, d2), _ = cv2.fitEllipse(contour)
                equiv_ellipse_area = np.pi * (d1 / 2.0) * (d2 / 2.0)
                asymmetry_fullness = equiv_ellipse_area / (total_area + eps)
            else:
                asymmetry_fullness = 1.0
        else:
            asymmetry_fullness = 0.0

        feature_rows.append([
            asymmetry_area_ratio,
            asymmetry_x_axis,
            asymmetry_xy_sum,
            asymmetry_rotation,
            asymmetry_fullness,
        ])

    return np.asarray(feature_rows, dtype=np.float32), feature_names


## 2.2 色彩特征

In [8]:
# functions for color features

def _safe_skew(vals: np.ndarray, eps: float = 1e-6) -> float:
    vals = np.asarray(vals, dtype=np.float32)
    if vals.size < 3 or np.nanstd(vals) < eps:
        return 0.0
    value = stats.skew(vals, nan_policy="omit")
    return float(0.0 if np.isnan(value) else value)


def _normalized_rgb(rgb: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    total = r + g + b + eps
    return np.stack([r / total, g / total, b / total], axis=-1).astype(np.float32)


def _ohta_color_space(rgb: np.ndarray) -> np.ndarray:
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return np.stack([
        (r + g + b) / 3.0,
        (r - b) / 2.0,
        (2.0 * g - r - b) / 4.0,
    ], axis=-1).astype(np.float32)


def _gevers_l123(rgb: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    total = r + g + b + eps
    return np.stack([
        ((r - g) ** 2) / total,
        ((r - b) ** 2) / total,
        ((g - b) ** 2) / total,
    ], axis=-1).astype(np.float32)


def _build_peripheral_regions(
    lesion_mask: np.ndarray,
    transition_ratio: float = 0.10,
    ring_ratio: float = 0.20,
) -> tuple[np.ndarray, np.ndarray]:
    outside_mask = ~lesion_mask
    lesion_area = int(lesion_mask.sum())

    if lesion_area <= 0 or outside_mask.sum() == 0:
        empty = np.zeros_like(lesion_mask, dtype=bool)
        return empty, empty

    dist_outside = cv2.distanceTransform(outside_mask.astype(np.uint8), cv2.DIST_L2, 5)
    outside_indices = np.argwhere(outside_mask)
    order = np.argsort(dist_outside[outside_mask])
    outside_indices_sorted = outside_indices[order]

    n_transition = int(round(transition_ratio * lesion_area))
    n_inner = int(round(ring_ratio * lesion_area))
    n_outer = int(round(ring_ratio * lesion_area))

    start_inner = min(n_transition, len(outside_indices_sorted))
    end_inner = min(start_inner + n_inner, len(outside_indices_sorted))
    end_outer = min(end_inner + n_outer, len(outside_indices_sorted))

    inner_mask = np.zeros_like(lesion_mask, dtype=bool)
    outer_mask = np.zeros_like(lesion_mask, dtype=bool)

    if end_inner > start_inner:
        inner_idx = outside_indices_sorted[start_inner:end_inner]
        inner_mask[inner_idx[:, 0], inner_idx[:, 1]] = True

    if end_outer > end_inner:
        outer_idx = outside_indices_sorted[end_inner:end_outer]
        outer_mask[outer_idx[:, 0], outer_idx[:, 1]] = True

    return inner_mask, outer_mask


def _region_mean_std(img3: np.ndarray, channel_names: Sequence[str], region_mask: np.ndarray) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    for idx, channel_name in enumerate(channel_names):
        vals = img3[:, :, idx][region_mask].astype(np.float32)
        if vals.size == 0:
            vals = img3[:, :, idx].reshape(-1).astype(np.float32)
        out[channel_name] = {
            "mean": float(np.mean(vals)),
            "std": float(np.std(vals)),
        }
    return out


def _extract_single_color_features(image: np.ndarray, mask: np.ndarray, eps: float = 1e-6) -> dict[str, float]:
    feats: dict[str, float] = {}
    img_rgb_uint8 = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    rgb = img_rgb_uint8.astype(np.float32)
    h_img, w_img = rgb.shape[:2]

    mask_binary = _to_binary_mask(mask)
    if mask_binary.shape[:2] != rgb.shape[:2]:
        mask_binary = cv2.resize(mask_binary, (w_img, h_img), interpolation=cv2.INTER_NEAREST)

    lesion_mask = mask_binary > 127
    if lesion_mask.sum() == 0:
        lesion_mask = np.ones((h_img, w_img), dtype=bool)

    norm_rgb = _normalized_rgb(rgb, eps=eps)
    hsv = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 0] *= 2.0
    ohta = _ohta_color_space(rgb)
    gevers = _gevers_l123(rgb, eps=eps)
    luv = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2Luv).astype(np.float32)
    lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2Lab).astype(np.float32)
    ycrcb = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2YCrCb).astype(np.float32)
    ycbcr = np.stack([ycrcb[:, :, 0], ycrcb[:, :, 2], ycrcb[:, :, 1]], axis=-1).astype(np.float32)

    color_spaces_all = {
        "RGB": (rgb, ["R", "G", "B"]),
        "rgb": (norm_rgb, ["r", "g", "b"]),
        "HSV": (hsv, ["H", "S", "V"]),
        "Ohta": (ohta, ["I1", "I2", "I3"]),
        "Gevers": (gevers, ["l1", "l2", "l3"]),
        "Luv": (luv, ["L_luv", "u_luv", "v_luv"]),
        "Lab": (lab, ["L_lab", "a_lab", "b_lab"]),
        "YCbCr": (ycbcr, ["Y", "Cb", "Cr"]),
    }

    for space_name, (space_img, channel_names) in color_spaces_all.items():
        for idx, channel_name in enumerate(channel_names):
            vals = space_img[:, :, idx][lesion_mask].astype(np.float32)
            if vals.size == 0:
                vals = space_img[:, :, idx].reshape(-1).astype(np.float32)
            feats[f"{space_name}_{channel_name}_max"] = float(np.max(vals))
            feats[f"{space_name}_{channel_name}_min"] = float(np.min(vals))
            feats[f"{space_name}_{channel_name}_mean"] = float(np.mean(vals))
            feats[f"{space_name}_{channel_name}_var"] = float(np.var(vals))
            feats[f"{space_name}_{channel_name}_std"] = float(np.std(vals))
            feats[f"{space_name}_{channel_name}_skew"] = _safe_skew(vals, eps=eps)

    inner_mask, outer_mask = _build_peripheral_regions(lesion_mask)
    regional_spaces = {
        "RGB": (rgb, ["R", "G", "B"]),
        "rgb": (norm_rgb, ["r", "g", "b"]),
        "HSV": (hsv, ["H", "S", "V"]),
        "Ohta": (ohta, ["I1", "I2", "I3"]),
        "Gevers": (gevers, ["l1", "l2", "l3"]),
        "Luv": (luv, ["L_luv", "u_luv", "v_luv"]),
    }

    for space_name, (space_img, channel_names) in regional_spaces.items():
        region_stats = {
            "lesion": _region_mean_std(space_img, channel_names, lesion_mask),
            "inner": _region_mean_std(space_img, channel_names, inner_mask),
            "outer": _region_mean_std(space_img, channel_names, outer_mask),
        }
        for channel_name in channel_names:
            for stat_name in ["mean", "std"]:
                for region_a, region_b in [("outer", "inner"), ("outer", "lesion"), ("inner", "lesion")]:
                    a = region_stats[region_a][channel_name][stat_name]
                    b = region_stats[region_b][channel_name][stat_name]
                    feats[f"{space_name}_{channel_name}_{stat_name}_{region_a}_div_{region_b}"] = float(a / (b + eps))
                    feats[f"{space_name}_{channel_name}_{stat_name}_{region_a}_minus_{region_b}"] = float(a - b)

    def add_color_asymmetry(channel_img: np.ndarray, channel_name: str) -> None:
        ys, xs = np.where(lesion_mask)
        vals = channel_img[lesion_mask].astype(np.float32)
        keys = [
            f"asym_{channel_name}_axis0_pct",
            f"asym_{channel_name}_axis0_sum",
            f"asym_{channel_name}_axis90_pct",
            f"asym_{channel_name}_axis90_sum",
        ]
        if vals.size < 5 or np.sum(vals) <= eps:
            for key in keys:
                feats[key] = 0.0
            return

        weights = vals + eps
        cx = np.sum(xs * weights) / np.sum(weights)
        cy = np.sum(ys * weights) / np.sum(weights)
        x0 = xs - cx
        y0 = ys - cy
        cov = np.array([
            [np.sum(weights * x0 * x0) / np.sum(weights), np.sum(weights * x0 * y0) / np.sum(weights)],
            [np.sum(weights * x0 * y0) / np.sum(weights), np.sum(weights * y0 * y0) / np.sum(weights)],
        ], dtype=np.float32)
        _, eigvecs = np.linalg.eigh(cov)
        ux, uy = eigvecs[:, -1]
        u = x0 * ux + y0 * uy
        v = -x0 * uy + y0 * ux

        def folded_difference(coord_a: np.ndarray, coord_b: np.ndarray, values: np.ndarray, fold_axis: str) -> tuple[float, float]:
            qa = np.round(coord_a).astype(int)
            qb = np.round(coord_b).astype(int)
            bins: dict[tuple[int, int], float] = {}
            for a, b, value in zip(qa, qb, values):
                key = (int(a), int(b))
                bins[key] = bins.get(key, 0.0) + float(value)
            visited: set[tuple[int, int]] = set()
            total_abs_diff = 0.0
            for key, value in bins.items():
                if key in visited:
                    continue
                a, b = key
                mirror_key = (-a, b) if fold_axis == "u" else (a, -b)
                total_abs_diff += abs(value - bins.get(mirror_key, 0.0))
                visited.add(key)
                visited.add(mirror_key)
            return float(total_abs_diff / (np.sum(values) + eps)), float(total_abs_diff / (len(values) + eps))

        feats[keys[0]], feats[keys[1]] = folded_difference(u, v, vals, fold_axis="u")
        feats[keys[2]], feats[keys[3]] = folded_difference(u, v, vals, fold_axis="v")

    add_color_asymmetry(rgb[:, :, 0], "R")
    add_color_asymmetry(rgb[:, :, 1], "G")
    add_color_asymmetry(rgb[:, :, 2], "B")

    ys, xs = np.where(lesion_mask)
    geom_cx = np.mean(xs)
    geom_cy = np.mean(ys)
    equiv_diameter = np.sqrt(4.0 * lesion_mask.sum() / np.pi) + eps
    for space_name, (space_img, channel_names) in regional_spaces.items():
        for idx, channel_name in enumerate(channel_names):
            vals = space_img[:, :, idx][lesion_mask].astype(np.float32)
            weights = vals - np.min(vals) + eps
            if np.sum(weights) <= eps:
                feats[f"centroid_dist_{space_name}_{channel_name}"] = 0.0
                continue
            bright_cx = np.sum(xs * weights) / np.sum(weights)
            bright_cy = np.sum(ys * weights) / np.sum(weights)
            dist = np.sqrt((bright_cx - geom_cx) ** 2 + (bright_cy - geom_cy) ** 2)
            feats[f"centroid_dist_{space_name}_{channel_name}"] = float(dist / equiv_diameter)

    def luv_hist(region_mask: np.ndarray) -> np.ndarray:
        vals = luv[region_mask]
        if vals.shape[0] == 0:
            return np.zeros(4 * 8 * 8, dtype=np.float32)
        hist, _ = np.histogramdd(vals, bins=(4, 8, 8), range=((0, 256), (0, 256), (0, 256)))
        hist = hist.astype(np.float32).reshape(-1)
        return hist / (hist.sum() + eps)

    hist_regions = {
        "lesion": luv_hist(lesion_mask),
        "inner": luv_hist(inner_mask),
        "outer": luv_hist(outer_mask),
    }
    for region_a, region_b in [("lesion", "inner"), ("lesion", "outer"), ("inner", "outer")]:
        hist_a = hist_regions[region_a]
        hist_b = hist_regions[region_b]
        feats[f"Luv_hist_L1_{region_a}_{region_b}"] = float(np.sum(np.abs(hist_a - hist_b)))
        feats[f"Luv_hist_L2_{region_a}_{region_b}"] = float(np.sqrt(np.sum((hist_a - hist_b) ** 2)))

    r = rgb[:, :, 0][lesion_mask].astype(np.float32)
    g = rgb[:, :, 1][lesion_mask].astype(np.float32)
    b = rgb[:, :, 2][lesion_mask].astype(np.float32)
    erythema_index = r / (g + b + eps)
    feats["erythema_index_mean"] = float(np.mean(erythema_index))
    feats["erythema_index_var"] = float(np.var(erythema_index))

    h_deg = hsv[:, :, 0][lesion_mask].astype(np.float32)
    red_hue_mask = ((h_deg >= 0) & (h_deg <= 20)) | ((h_deg >= 340) & (h_deg <= 360))
    feats["hue_red_ratio"] = float(np.mean(red_hue_mask))

    return feats


def extract_lesion_feature_color(
    images: Sequence[np.ndarray],
    masks: Sequence[np.ndarray],
) -> tuple[np.ndarray, list[str]]:
    """Extract color features and return a matrix plus feature names."""
    feature_dicts = [_extract_single_color_features(image, mask) for image, mask in zip(images, masks)]
    feature_names = list(feature_dicts[0].keys()) if feature_dicts else []
    feature_rows = [[feature_dict.get(name, 0.0) for name in feature_names] for feature_dict in feature_dicts]
    return np.asarray(feature_rows, dtype=np.float32), feature_names


## 2.3 纹理特征：GLCM

In [9]:
# functions for texture features: GLCM

_GLCM_FEATURE_NAMES = [
    "glcm_ASM",
    "glcm_Contrast",
    "glcm_Correlation",
    "glcm_Homogeneity",
    "glcm_Dissimilarity",
    "glcm_Entropy",
    "glcm_MaxProbability",
    "glcm_Variance",
    "glcm_SumVariance",
    "glcm_SumEntropy",
    "glcm_DifferenceVariance",
    "glcm_DifferenceEntropy",
    "glcm_IMCorr1",
    "glcm_IMCorr2",
]


def _compute_single_glcm_features(
    image: np.ndarray,
    mask: np.ndarray,
    distances: Sequence[int] = (1,),
    angles: Sequence[float] = (0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    levels: int = 64,
) -> list[float]:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    binary_mask = _to_binary_mask(mask)
    if binary_mask.shape[:2] != gray.shape[:2]:
        binary_mask = cv2.resize(binary_mask, (gray.shape[1], gray.shape[0]), interpolation=cv2.INTER_NEAREST)

    coords = cv2.findNonZero(binary_mask)
    if coords is None:
        return [np.nan] * len(_GLCM_FEATURE_NAMES)

    x, y, w, h = cv2.boundingRect(coords)
    roi_gray = gray[y:y + h, x:x + w]
    roi_gray = (roi_gray / 256 * levels).astype(np.uint8)
    roi_gray[roi_gray >= levels] = levels - 1

    glcm = graycomatrix(
        roi_gray,
        distances=list(distances),
        angles=list(angles),
        levels=levels,
        symmetric=True,
        normed=True,
    )

    values_by_feature = {name: [] for name in _GLCM_FEATURE_NAMES}

    for angle_idx in range(len(angles)):
        glcm_angle = glcm[:, :, 0, angle_idx]
        glcm_4d = glcm_angle.reshape(levels, levels, 1, 1)

        values_by_feature["glcm_ASM"].append(graycoprops(glcm_4d, "ASM")[0, 0])
        values_by_feature["glcm_Contrast"].append(graycoprops(glcm_4d, "contrast")[0, 0])
        values_by_feature["glcm_Correlation"].append(graycoprops(glcm_4d, "correlation")[0, 0])
        values_by_feature["glcm_Homogeneity"].append(graycoprops(glcm_4d, "homogeneity")[0, 0])
        values_by_feature["glcm_Dissimilarity"].append(graycoprops(glcm_4d, "dissimilarity")[0, 0])

        p = glcm_angle + 1e-12
        values_by_feature["glcm_Entropy"].append(-np.sum(p * np.log(p)))
        values_by_feature["glcm_MaxProbability"].append(np.max(glcm_angle))

        px = np.sum(glcm_angle, axis=1)
        py = np.sum(glcm_angle, axis=0)
        mu = np.sum(np.arange(levels) * px)
        values_by_feature["glcm_Variance"].append(np.sum((np.arange(levels)[:, None] - mu) ** 2 * glcm_angle))

        sum_prob = np.zeros(2 * levels + 1)
        diff_prob = np.zeros(levels)
        for i_idx in range(levels):
            for j_idx in range(levels):
                prob = glcm_angle[i_idx, j_idx]
                sum_prob[i_idx + j_idx] += prob
                diff_prob[abs(i_idx - j_idx)] += prob

        k_vals = np.arange(2, 2 * levels + 1)
        sum_prob_valid = sum_prob[2:2 * levels + 1]
        mu_sum = np.sum(k_vals * sum_prob_valid)
        values_by_feature["glcm_SumVariance"].append(np.sum((k_vals - mu_sum) ** 2 * sum_prob_valid))
        values_by_feature["glcm_SumEntropy"].append(-np.sum(sum_prob_valid * np.log(sum_prob_valid + 1e-12)))

        mu_diff = np.sum(np.arange(levels) * diff_prob)
        values_by_feature["glcm_DifferenceVariance"].append(np.sum((np.arange(levels) - mu_diff) ** 2 * diff_prob))
        values_by_feature["glcm_DifferenceEntropy"].append(-np.sum(diff_prob * np.log(diff_prob + 1e-12)))

        hx = -np.sum(px * np.log(px + 1e-12))
        hy = -np.sum(py * np.log(py + 1e-12))
        hxy = -np.sum(glcm_angle * np.log(glcm_angle + 1e-12))
        hxy1 = -np.sum(glcm_angle * np.log(px[:, None] * py[None, :] + 1e-12))
        hxy2 = -np.sum(px[:, None] * py[None, :] * np.log(px[:, None] * py[None, :] + 1e-12))
        values_by_feature["glcm_IMCorr1"].append((hxy - hxy1) / (max(hx, hy) + 1e-12))
        values_by_feature["glcm_IMCorr2"].append(np.sqrt(max(0.0, 1 - np.exp(-2 * (hxy2 - hxy)))))

    return [float(np.mean(values_by_feature[name])) for name in _GLCM_FEATURE_NAMES]


def extract_lesion_feature_glcm(
    images: Sequence[np.ndarray],
    masks: Sequence[np.ndarray],
    distances: Sequence[int] = (1,),
    angles: Sequence[float] = (0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    levels: int = 64,
) -> tuple[np.ndarray, list[str]]:
    """Extract GLCM texture features and return a matrix plus feature names."""
    feature_rows = [
        _compute_single_glcm_features(
            image,
            mask,
            distances=distances,
            angles=angles,
            levels=levels,
        )
        for image, mask in zip(images, masks)
    ]
    return np.asarray(feature_rows, dtype=np.float32), _GLCM_FEATURE_NAMES.copy()


## 2.4 辅助函数

- 整合特征
- 整合特征名字列表
- 展示特征信息

In [10]:
# functions for concat features and feature names

def concat_features(*feature_matrices: np.ndarray | Sequence[np.ndarray]) -> np.ndarray:
    """Concatenate feature matrices by columns."""
    if len(feature_matrices) == 1 and isinstance(feature_matrices[0], (list, tuple)):
        feature_matrices = tuple(feature_matrices[0])
    return np.concatenate(feature_matrices, axis=1)


def concat_feature_names(*feature_name_lists: Sequence[str] | Sequence[Sequence[str]]) -> list[str]:
    """Concatenate feature-name lists while preserving order."""
    if len(feature_name_lists) == 1 and isinstance(feature_name_lists[0], (list, tuple)):
        feature_name_lists = tuple(feature_name_lists[0])
    names: list[str] = []
    for feature_name_list in feature_name_lists:
        names.extend(feature_name_list)
    return names

In [11]:
# function for showing the features information

def show_features_information(
    features: np.ndarray,
    feature_names: Sequence[str],
    title: str = "Features information",
    preview_rows: int = 12,
) -> pd.DataFrame:
    """Print a compact summary and return a per-feature information table."""
    features_array = np.asarray(features, dtype=np.float32)
    info_df = pd.DataFrame({
        "feature": list(feature_names),
        "mean": np.nanmean(features_array, axis=0),
        "std": np.nanstd(features_array, axis=0),
        "min": np.nanmin(features_array, axis=0),
        "max": np.nanmax(features_array, axis=0),
        "nan_count": np.isnan(features_array).sum(axis=0),
    })

    print("=" * 70)
    print(title)
    print("=" * 70)
    print(f"samples: {features_array.shape[0]}")
    print(f"features: {features_array.shape[1]}")
    print(f"missing values: {int(np.isnan(features_array).sum())}")
    print("\nFeature preview:")

    try:
        display(info_df.head(preview_rows))
    except NameError:
        print(info_df.head(preview_rows).to_string(index=False))

    return info_df


## 2.5 Test

In [12]:
# test cell
PROJECT_DIR = Path('./')
IMAGE_DIR = PROJECT_DIR / 'image_processed'
MASK_DIR = PROJECT_DIR / 'mask'

image_paths = sorted(IMAGE_DIR.glob('*.jpg'))
images: list[np.ndarray] = []
masks: list[np.ndarray] = []
filenames: list[str] = []

for image_path in image_paths:
    mask_path = MASK_DIR / f'mask_{image_path.name}'
    if not mask_path.exists():
        continue

    image = cv2.imread(str(image_path))
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None:
        continue

    images.append(image)
    masks.append(mask)
    filenames.append(image_path.name)

print(f'Loaded {len(images)} image-mask pairs.')

shape_features, shape_feature_names = extract_lesion_feature_shape(images, masks)
asymmetry_features, asymmetry_feature_names = extract_lesion_feature_asymmetry(images, masks)
color_features, color_feature_names = extract_lesion_feature_color(images, masks)
glcm_features, glcm_feature_names = extract_lesion_feature_glcm(images, masks)

all_features = concat_features(
    shape_features,
    asymmetry_features,
    color_features,
    glcm_features,
)
all_feature_names = concat_feature_names(
    shape_feature_names,
    asymmetry_feature_names,
    color_feature_names,
    glcm_feature_names,
)

show_features_information(shape_features, shape_feature_names, title='Shape features')
show_features_information(asymmetry_features, asymmetry_feature_names, title='Asymmetry features')
show_features_information(color_features, color_feature_names, title='Color features')
show_features_information(glcm_features, glcm_feature_names, title='GLCM texture features')
all_feature_info = show_features_information(all_features, all_feature_names, title='All extracted features')

feature_df = pd.DataFrame(all_features, columns=all_feature_names)
feature_df.insert(0, 'filename', filenames)
feature_df.insert(1, 'image_id', [Path(name).stem for name in filenames])
feature_df.insert(2, 'base_id', [Path(name).stem.split('_aug')[0] for name in filenames])

try:
    display(feature_df.head())
except NameError:
    print(feature_df.head().to_string(index=False))


Loaded 600 image-mask pairs.
Shape features
samples: 600
features: 12
missing values: 0

Feature preview:


,feature,mean,std,min,max,nan_count
0,shape_area,11766.531250,10025.583008,367.000000,44782.000000,0
1,shape_perimeter,422.184174,211.651169,79.941124,906.264038,0
2,shape_circularity,0.708923,0.098203,0.317197,0.881249,0
3,shape_max_diameter,135.805359,63.641773,27.802877,289.401093,0
4,shape_equiv_diameter,110.731956,52.153942,21.616634,238.784866,0
5,shape_aspect_ratio,1.386049,0.265180,1.005117,2.576981,0
6,shape_eccentricity,0.638031,0.151213,0.100780,0.921638,0
7,shape_compactness,0.818130,0.065255,0.626937,0.950987,0
8,shape_solidity,0.948471,0.032426,0.815156,0.992053,0
9,shape_rectangularity,0.755236,0.041609,0.620231,0.900521,0


Asymmetry features
samples: 600
features: 5
missing values: 0

Feature preview:


,feature,mean,std,min,max,nan_count
0,asymmetry_area_ratio,2.308360,1.885632,0.007843,12.106455,0
1,asymmetry_x_axis,53.671070,39.370342,5.283789,200.000000,0
2,asymmetry_xy_sum,107.676964,67.509399,9.751120,400.000000,0
3,asymmetry_rotation,0.838536,0.082290,0.500956,0.955533,0
4,asymmetry_fullness,1.014652,0.068938,0.909649,1.780739,0


Color features
samples: 600
features: 399
missing values: 0

Feature preview:


...

GLCM texture features
samples: 600
features: 14
missing values: 0

Feature preview:


,feature,mean,std,min,max,nan_count
0,glcm_ASM,0.013932,0.008945,0.003983,0.125363,0
1,glcm_Contrast,3.287185,4.011892,0.405938,43.085350,0
2,glcm_Correlation,0.984871,0.011844,0.905160,0.997125,0
3,glcm_Homogeneity,0.640366,0.077071,0.375867,0.825640,0
4,glcm_Dissimilarity,0.974428,0.415656,0.367280,3.782364,0
5,glcm_Entropy,4.917315,0.449569,3.250856,6.066134,0
6,glcm_MaxProbability,0.045181,0.024957,0.011273,0.345347,0
7,glcm_Variance,118.676704,74.396927,4.691899,362.890015,0
8,glcm_SumVariance,450.170990,289.300842,18.200188,1442.653687,0
9,glcm_SumEntropy,4.049987,0.291210,2.829751,4.675649,0


All extracted features
samples: 600
features: 430
missing values: 0

Feature preview:


,feature,mean,std,min,max,nan_count
0,shape_area,11766.531250,10025.583008,367.000000,44782.000000,0
1,shape_perimeter,422.184174,211.651169,79.941124,906.264038,0
2,shape_circularity,0.708923,0.098203,0.317197,0.881249,0
3,shape_max_diameter,135.805359,63.641773,27.802877,289.401093,0
4,shape_equiv_diameter,110.731956,52.153942,21.616634,238.784866,0
5,shape_aspect_ratio,1.386049,0.265180,1.005117,2.576981,0
6,shape_eccentricity,0.638031,0.151213,0.100780,0.921638,0
7,shape_compactness,0.818130,0.065255,0.626937,0.950987,0
8,shape_solidity,0.948471,0.032426,0.815156,0.992053,0
9,shape_rectangularity,0.755236,0.041609,0.620231,0.900521,0


,filename,image_id,base_id,shape_area,shape_perimeter,shape_circularity,shape_max_diameter,shape_equiv_diameter,shape_aspect_ratio,shape_eccentricity,...,glcm_Dissimilarity,glcm_Entropy,glcm_MaxProbability,glcm_Variance,glcm_SumVariance,glcm_SumEntropy,glcm_DifferenceVariance,glcm_DifferenceEntropy,glcm_IMCorr1,glcm_IMCorr2
0,1.jpg,1,1,7530.5,348.291412,0.780096,117.175079,97.918999,1.398464,0.699052,...,0.679325,4.589345,0.059139,64.824448,258.213562,3.931899,0.611229,1.052142,-0.605086,0.990366
1,10.jpg,10,10,29859.5,714.173645,0.735673,230.288956,194.982803,1.082319,0.382533,...,0.410149,4.107319,0.054348,40.252808,160.511124,3.740886,0.327356,0.778750,-0.679859,0.992548
2,100.jpg,100,100,21472.0,632.641724,0.674166,218.442215,165.345093,1.505464,0.747513,...,0.944507,5.270572,0.016135,90.222115,358.826721,4.300153,1.150154,1.252803,-0.542524,0.989731
3,100_aug1.jpg,100_aug1,100,21472.0,632.641724,0.674166,218.442215,165.345093,1.505464,0.747513,...,0.953145,5.279065,0.015844,90.171318,358.613495,4.301238,1.144164,1.257866,-0.540537,0.989599
4,100_aug2.jpg,100_aug2,100,21396.5,665.997009,0.606189,212.362900,165.054138,1.404410,0.702136,...,0.940626,5.226100,0.016482,84.021080,324.490906,4.266317,1.628538,1.238001,-0.544200,0.989558


# 3 学习分类

以下用基础机器学习模型实现分类：
1. SVM
2. 随机森林
3. XGBoost

## 3.0 辅助函数

- 划分训练集 & 测试集
- 获取 label

In [13]:

def prepare_feature_matrix(
    features: np.ndarray,
    fill_value: float = 0.0,
) -> np.ndarray:
    """Convert feature values to a finite float32 matrix for model training."""
    return np.nan_to_num(
        np.asarray(features, dtype=np.float32),
        nan=fill_value,
        posinf=fill_value,
        neginf=fill_value,
    )


# fucntions for Spliting the Training and Testing sets
def split_classification_dataset(
    features: np.ndarray,
    labels: Sequence[str],
    test_size: float = 0.3,
    random_state: int = 42,
    stratify: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Split features and labels into training and testing sets."""
    x = prepare_feature_matrix(features)
    y = np.asarray(labels)
    stratify_labels = y if stratify else None

    return train_test_split(
        x,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_labels,
    )

# function for attaching labels from label.csv to the feature DataFrame
def attach_labels_from_csv(
    feature_df: pd.DataFrame,
    label_csv: str | Path,
    image_id_column: str = "base_id",
    label_column: str = "dx",
) -> pd.DataFrame:
    """Attach class labels from label.csv to a feature DataFrame."""
    labels_df = pd.read_csv(label_csv)
    labels_df["image_id"] = labels_df["image_id"].astype(str)
    labels_df = labels_df.rename(columns={"image_id": image_id_column, label_column: "label"})

    output_df = feature_df.copy()
    output_df[image_id_column] = output_df[image_id_column].astype(str)
    return output_df.merge(labels_df[[image_id_column, "label"]], on=image_id_column, how="left")


## 3.1 SVM

### 3.1.1 创建 & 训练

In [16]:
# functions for building SVM model and training the model

def build_svm_classifier(
    c: float = 1.0,
    kernel: str = "rbf",
    gamma: str | float = "scale",
    class_weight: str | dict[str, float] | None = "balanced",
    probability: bool = True,
    random_state: int = 42,
) -> Pipeline:
    """Create a standard scaler + SVM classifier pipeline."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=c,
            kernel=kernel,
            gamma=gamma,
            class_weight=class_weight,
            probability=probability,
            random_state=random_state,
        )),
    ])


def train_svm_model(
    x_train: np.ndarray,
    y_train: Sequence[str],
    c: float = 1.0,
    kernel: str = "rbf",
    gamma: str | float = "scale",
    class_weight: str | dict[str, float] | None = "balanced",
    probability: bool = True,
    random_state: int = 42,
) -> Pipeline:
    """Build and train an SVM classifier."""
    model = build_svm_classifier(
        c=c,
        kernel=kernel,
        gamma=gamma,
        class_weight=class_weight,
        probability=probability,
        random_state=random_state,
    )
    model.fit(prepare_feature_matrix(x_train), np.asarray(y_train))
    return model


def run_svm_training_pipeline(
    features: np.ndarray,
    labels: Sequence[str],
    feature_names: Sequence[str],
    test_size: float = 0.3,
    split_random_state: int = 42,
    model_random_state: int = 42,
    c: float = 1.0,
    kernel: str = "rbf",
    gamma: str | float = "scale",
) -> dict[str, object]:
    """Split data, train SVM, and return model plus held-out data."""
    x_train, x_test, y_train, y_test = split_classification_dataset(
        features,
        labels,
        test_size=test_size,
        random_state=split_random_state,
        stratify=True,
    )
    model = train_svm_model(
        x_train,
        y_train,
        c=c,
        kernel=kernel,
        gamma=gamma,
        random_state=model_random_state,
    )
    return {
        "model": model,
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": list(feature_names),
    }


### 3.1.2 预测 & 评估

In [17]:
# functions for pridicting and evaluating the model

def evaluate_svm_model(
    model: Pipeline,
    x_test: np.ndarray,
    y_test: Sequence[str],
    label_order: Sequence[str] | None = None,
) -> dict[str, object]:
    """Predict test labels and compute hit rate, confusion matrix, and report."""
    y_true = np.asarray(y_test)
    y_pred = model.predict(prepare_feature_matrix(x_test))
    labels = list(label_order) if label_order is not None else sorted(np.unique(y_true).tolist())

    hit_rate = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
        output_dict=True,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )

    return {
        "hit_rate": float(hit_rate),
        "confusion_matrix": cm,
        "classification_report": report_text,
        "classification_report_dict": report_dict,
        "labels": labels,
        "y_pred": y_pred,
    }


def print_svm_evaluation(evaluation: dict[str, object]) -> None:
    """Print SVM evaluation results in a compact form."""
    labels = evaluation["labels"]
    cm_df = pd.DataFrame(
        evaluation["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

    print("=" * 70)
    print("SVM evaluation")
    print("=" * 70)
    print(f"Hit rate: {evaluation['hit_rate']:.4f}")
    print("\nConfusion matrix:")
    try:
        display(cm_df)
    except NameError:
        print(cm_df.to_string())
    print("\nClassification report:")
    print(evaluation["classification_report"])


### 3.1.3 自动优化超参数

In [18]:
# functions for optimizing the hyperparameters of the model

def get_default_svm_param_grid() -> dict[str, list[object]]:
    """Return a compact SVM hyperparameter search grid."""
    return {
        "clf__kernel": ["rbf", "linear", "poly"],
        "clf__C": [0.1, 1.0, 10.0, 100.0],
        "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    }


def optimize_svm_hyperparameters(
    features: np.ndarray,
    labels: Sequence[str],
    param_grid: dict[str, list[object]] | None = None,
    cv: int = 5,
    scoring: str = "accuracy",
    random_state: int = 42,
    n_jobs: int = -1,
) -> dict[str, object]:
    """Optimize SVM hyperparameters with grid search."""
    x = prepare_feature_matrix(features)
    y = np.asarray(labels)
    param_grid = get_default_svm_param_grid() if param_grid is None else param_grid

    base_model = build_svm_classifier(random_state=random_state)
    splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring=scoring,
        cv=splitter,
        n_jobs=n_jobs,
        refit=True,
        verbose=1,
    )
    search.fit(x, y)

    return {
        "search": search,
        "best_model": search.best_estimator_,
        "best_params": search.best_params_,
        "best_cv_score": float(search.best_score_),
        "cv_results": pd.DataFrame(search.cv_results_),
    }


def print_svm_optimization_result(optimization_result: dict[str, object]) -> None:
    """Print the best SVM hyperparameters and cross-validation score."""
    print("=" * 70)
    print("SVM hyperparameter optimization")
    print("=" * 70)
    print(f"Best CV score: {optimization_result['best_cv_score']:.4f}")
    print("Best parameters:")
    for key, value in optimization_result["best_params"].items():
        print(f"  {key}: {value}")


### 3.1.4 Test

In [19]:
# test cell
LABEL_CSV = Path('./label.csv')

labeled_feature_df = attach_labels_from_csv(feature_df, LABEL_CSV)
model_rows = labeled_feature_df['label'].notna().values
svm_features = labeled_feature_df.loc[model_rows, all_feature_names].values
svm_labels = labeled_feature_df.loc[model_rows, 'label'].values

svm_result = run_svm_training_pipeline(
    features=svm_features,
    labels=svm_labels,
    feature_names=all_feature_names,
    test_size=0.3,
    split_random_state=42,
    model_random_state=42,
    c=1.0,
    kernel='rbf',
    gamma='scale',
)

svm_evaluation = evaluate_svm_model(
    svm_result['model'],
    svm_result['x_test'],
    svm_result['y_test'],
    label_order=['mel', 'nv', 'vasc'],
)
print_svm_evaluation(svm_evaluation)

# Optional hyperparameter optimization. This can take longer on all 430 features.
# svm_optimization = optimize_svm_hyperparameters(
#     svm_features,
#     svm_labels,
#     cv=5,
#     scoring='accuracy',
# )
# print_svm_optimization_result(svm_optimization)


SVM evaluation
Hit rate: 0.8611

Confusion matrix:


,pred_mel,pred_nv,pred_vasc
true_mel,54,9,0
true_nv,13,74,3
true_vasc,0,0,27



Classification report:
              precision    recall  f1-score   support

         mel       0.81      0.86      0.83        63
          nv       0.89      0.82      0.86        90
        vasc       0.90      1.00      0.95        27

    accuracy                           0.86       180
   macro avg       0.87      0.89      0.88       180
weighted avg       0.86      0.86      0.86       180



## 3.2 随机森林

### 3.2.1 创建 & 训练

In [22]:
# functions for building Random Forest model and training the model

def build_random_forest_classifier(
    n_estimators: int = 500,
    max_depth: int | None = None,
    min_samples_leaf: int = 2,
    min_samples_split: int = 2,
    max_features: str | int | float | None = "sqrt",
    class_weight: str | dict[str, float] | None = "balanced",
    random_state: int = 42,
    n_jobs: int = -1,
) -> RandomForestClassifier:
    """Create a Random Forest classifier."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        class_weight=class_weight,
        random_state=random_state,
        n_jobs=n_jobs,
    )


def train_random_forest_model(
    x_train: np.ndarray,
    y_train: Sequence[str],
    n_estimators: int = 500,
    max_depth: int | None = None,
    min_samples_leaf: int = 2,
    min_samples_split: int = 2,
    max_features: str | int | float | None = "sqrt",
    class_weight: str | dict[str, float] | None = "balanced",
    random_state: int = 42,
    n_jobs: int = -1,
) -> RandomForestClassifier:
    """Build and train a Random Forest classifier."""
    model = build_random_forest_classifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        class_weight=class_weight,
        random_state=random_state,
        n_jobs=n_jobs,
    )
    model.fit(prepare_feature_matrix(x_train), np.asarray(y_train))
    return model


def get_random_forest_feature_importance(
    model: RandomForestClassifier,
    feature_names: Sequence[str],
) -> list[tuple[str, float]]:
    """Return feature importances sorted from high to low."""
    importance_pairs = [
        (feature_name, float(importance))
        for feature_name, importance in zip(feature_names, model.feature_importances_)
    ]
    return sorted(importance_pairs, key=lambda item: item[1], reverse=True)


def run_random_forest_training_pipeline(
    features: np.ndarray,
    labels: Sequence[str],
    feature_names: Sequence[str],
    test_size: float = 0.3,
    split_random_state: int = 42,
    model_random_state: int = 42,
    n_estimators: int = 500,
    max_depth: int | None = None,
    min_samples_leaf: int = 2,
    min_samples_split: int = 2,
    max_features: str | int | float | None = "sqrt",
) -> dict[str, object]:
    """Split data, train Random Forest, and return model plus held-out data."""
    x_train, x_test, y_train, y_test = split_classification_dataset(
        features,
        labels,
        test_size=test_size,
        random_state=split_random_state,
        stratify=True,
    )
    model = train_random_forest_model(
        x_train,
        y_train,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=model_random_state,
    )
    return {
        "model": model,
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": list(feature_names),
        "feature_importances": get_random_forest_feature_importance(model, feature_names),
    }


### 3.2.2 预测 & 评估

In [23]:
# functions for predicting and evaluating the Random Forest model

def evaluate_random_forest_model(
    model: RandomForestClassifier,
    x_test: np.ndarray,
    y_test: Sequence[str],
    label_order: Sequence[str] | None = None,
) -> dict[str, object]:
    """Predict test labels and compute hit rate, confusion matrix, and report."""
    y_true = np.asarray(y_test)
    y_pred = model.predict(prepare_feature_matrix(x_test))
    labels = list(label_order) if label_order is not None else sorted(np.unique(y_true).tolist())

    hit_rate = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
        output_dict=True,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )

    return {
        "hit_rate": float(hit_rate),
        "confusion_matrix": cm,
        "classification_report": report_text,
        "classification_report_dict": report_dict,
        "labels": labels,
        "y_pred": y_pred,
    }


def print_random_forest_evaluation(evaluation: dict[str, object]) -> None:
    """Print Random Forest evaluation results in a compact form."""
    labels = evaluation["labels"]
    cm_df = pd.DataFrame(
        evaluation["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

    print("=" * 70)
    print("Random Forest evaluation")
    print("=" * 70)
    print(f"Hit rate: {evaluation['hit_rate']:.4f}")
    print("\nConfusion matrix:")
    try:
        display(cm_df)
    except NameError:
        print(cm_df.to_string())
    print("\nClassification report:")
    print(evaluation["classification_report"])


def print_top_random_forest_features(
    feature_importances: Sequence[tuple[str, float]],
    top_n: int = 20,
) -> None:
    """Print top Random Forest feature importances."""
    print(f"Top {min(top_n, len(feature_importances))} Random Forest feature importances:")
    for feature_name, importance in feature_importances[:top_n]:
        print(f"{feature_name}: {importance:.6f}")


### 3.2.3 自动优化超参数

In [24]:
# functions for optimizing the hyperparameters of the Random Forest model

def get_default_random_forest_param_grid() -> dict[str, list[object]]:
    """Return a compact Random Forest hyperparameter search grid."""
    return {
        "n_estimators": [100, 300, 500],
        "max_depth": [None, 8, 16, 24],
        "min_samples_leaf": [1, 2, 4],
        "min_samples_split": [2, 5, 10],
        "max_features": ["sqrt", "log2", 0.5],
    }


def optimize_random_forest_hyperparameters(
    features: np.ndarray,
    labels: Sequence[str],
    param_grid: dict[str, list[object]] | None = None,
    cv: int = 5,
    scoring: str = "accuracy",
    random_state: int = 42,
    n_jobs: int = -1,
) -> dict[str, object]:
    """Optimize Random Forest hyperparameters with grid search."""
    x = prepare_feature_matrix(features)
    y = np.asarray(labels)
    param_grid = get_default_random_forest_param_grid() if param_grid is None else param_grid

    base_model = build_random_forest_classifier(random_state=random_state)
    splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring=scoring,
        cv=splitter,
        n_jobs=n_jobs,
        refit=True,
        verbose=1,
    )
    search.fit(x, y)

    return {
        "search": search,
        "best_model": search.best_estimator_,
        "best_params": search.best_params_,
        "best_cv_score": float(search.best_score_),
        "cv_results": pd.DataFrame(search.cv_results_),
    }


def print_random_forest_optimization_result(optimization_result: dict[str, object]) -> None:
    """Print the best Random Forest hyperparameters and cross-validation score."""
    print("=" * 70)
    print("Random Forest hyperparameter optimization")
    print("=" * 70)
    print(f"Best CV score: {optimization_result['best_cv_score']:.4f}")
    print("Best parameters:")
    for key, value in optimization_result["best_params"].items():
        print(f"  {key}: {value}")


### 3.2.4 Test

In [25]:
# test cell
LABEL_CSV = Path('./label.csv')

labeled_feature_df = attach_labels_from_csv(feature_df, LABEL_CSV)
model_rows = labeled_feature_df['label'].notna().values
rf_features = labeled_feature_df.loc[model_rows, all_feature_names].values
rf_labels = labeled_feature_df.loc[model_rows, 'label'].values

rf_result = run_random_forest_training_pipeline(
    features=rf_features,
    labels=rf_labels,
    feature_names=all_feature_names,
    test_size=0.3,
    split_random_state=42,
    model_random_state=42,
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=2,
    max_features='sqrt',
)

rf_evaluation = evaluate_random_forest_model(
    rf_result['model'],
    rf_result['x_test'],
    rf_result['y_test'],
    label_order=['mel', 'nv', 'vasc'],
)
print_random_forest_evaluation(rf_evaluation)
print()
print_top_random_forest_features(rf_result['feature_importances'], top_n=20)

# Optional hyperparameter optimization. This can take longer on all 430 features.
# rf_optimization = optimize_random_forest_hyperparameters(
#     rf_features,
#     rf_labels,
#     cv=5,
#     scoring='accuracy',
# )
# print_random_forest_optimization_result(rf_optimization)


Random Forest evaluation
Hit rate: 0.9444

Confusion matrix:


,pred_mel,pred_nv,pred_vasc
true_mel,60,3,0
true_nv,3,84,3
true_vasc,0,1,26



Classification report:
              precision    recall  f1-score   support

         mel       0.95      0.95      0.95        63
          nv       0.95      0.93      0.94        90
        vasc       0.90      0.96      0.93        27

    accuracy                           0.94       180
   macro avg       0.93      0.95      0.94       180
weighted avg       0.95      0.94      0.94       180


Top 20 Random Forest feature importances:
HSV_H_mean: 0.027809
YCbCr_Cb_min: 0.022825
Luv_v_luv_max: 0.021725
Luv_v_luv_mean_outer_div_lesion: 0.018430
Luv_v_luv_mean_outer_minus_lesion: 0.018193
YCbCr_Cb_mean: 0.018038
Luv_v_luv_mean_inner_minus_lesion: 0.017058
Luv_v_luv_min: 0.016929
Luv_v_luv_mean: 0.016665
Luv_v_luv_mean_inner_div_lesion: 0.015389
Ohta_I3_min: 0.015061
Lab_b_lab_mean: 0.014936
Lab_b_lab_max: 0.014782
shape_solidity: 0.013933
asymmetry_rotation: 0.013812
rgb_b_min: 0.012700
asymmetry_fullness: 0.011913
shape_defect_ratio: 0.010815
rgb_b_mean: 0.009844
rgb_b_mean_inne